# Overview of JAZ

## Setup

Install JAZ, provide an API key, and configure a model. Load the data used in this
notebook's example ([Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/))
and set up evaluation.

In [7]:
%pip install -q jaz-lang pandas scipy

Note: you may need to restart the kernel to use updated packages.


In [16]:
import os, getpass

import jaz
from jaz.llm import LiteLLM
from jaz.repl import PythonREPL
from jaz.hooks import Hook

# if not os.environ.get('OPENAI_API_KEY'):
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

# Subagents usually need stronger models -- here we use gpt-5.4
# Subagents in this tutorial
# Subagents can take a few minutes in total, longer than the REPL's default limit of
# 30 seconds per code execution, so we lift that limit.
jaz.configure(llm=LiteLLM(model='openai/gpt-5.4'), repl=PythonREPL(exec_timeout=None, allowed_imports=['pandas', 're', 'numpy', 'concurrent'], allow_raise=False))


def abbreviate(text: str, prefix_len: int | None=None, suffix_len: int | None=None) -> str:
    """Abbreviates the text to first 5 and last 5 lines."""
    lines = text.split("\n")
    if prefix_len is None and suffix_len is None:
        return text
    prefix_len = prefix_len or 0
    suffix_len = suffix_len or 0
    if len(lines) > prefix_len + suffix_len:
        return "\n".join(lines[:prefix_len] + [f"[...{len(lines) - prefix_len - suffix_len} lines omitted...]"] + lines[-suffix_len:])
    return text

class PrintTrajectory(Hook):
    """Prints the prompt and agent trajectory."""

    def __init__(self, abbrev_prefix_len: int | None=None, abbrev_suffix_len: int | None=None):
        self.abbrev_prefix_len = abbrev_prefix_len
        self.abbrev_suffix_len = abbrev_suffix_len
        super().__init__()

    def on_llm_query_enter(self, event):
        if event.iteration > 0:
            print(f"--- REPL OUTPUT BEGINS ---\n{abbreviate(event.messages[-1]['content'], self.abbrev_prefix_len, self.abbrev_suffix_len)}\n--- REPL OUTPUT ENDS ---")

    def on_llm_query_complete(self, event):
        print(f"--- LLM RESPONSE BEGINS ---\n{abbreviate(event.response.content, self.abbrev_prefix_len, self.abbrev_suffix_len)}\n--- LLM RESPONSE ENDS ---")

In [17]:
from pathlib import Path
import pandas as pd

# The notes are in this repository, next to the notebook; on Colab, we fetch them from GitHub.
NOTES = "data/penguin_field_notes.csv"
if not Path(NOTES).exists():
    NOTES = "https://raw.githubusercontent.com/jaz-lang/cookbook/main/notebooks/" + NOTES
notes = pd.read_csv(NOTES)

# Print every note in full, and DataFrames with all their columns on one line.
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)

# The notes describe the 344 penguins of Palmer Penguins, in order: our ground truth.
penguins = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv")
NUMERIC = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
# Weights given in lb come from a spring scale that reads 0.2 lb heavy.
IN_LB = notes.note.str.contains(r"\d lb")
# Notes in inches and lb are rounded to 2 decimals, so allow for that rounding.
TOLERANCE = {"bill_length_mm": 0.2, "bill_depth_mm": 0.2, "flipper_length_mm": 0.2, "body_mass_g": 5}


def _is_correct(got, truth, column: str) -> bool:
    try:
        if pd.isna(truth):
            return got is None or pd.isna(got)
        if column in NUMERIC:
            return abs(float(got) - truth) <= TOLERANCE[column]
        return got == truth
    except (TypeError, ValueError):
        return False


def score(table: pd.DataFrame) -> dict:
    """How many cells of the transcribed `table` match the original data."""
    if len(table) != len(penguins):
        return {"error": f"{len(table)} rows instead of {len(penguins)}"}
    table = table.reset_index(drop=True)
    cells = [(i, col) for i in penguins.index for col in penguins.columns]
    right = {
        (i, col) for i, col in cells
        if col in table.columns and _is_correct(table[col].iloc[i], penguins.at[i, col], col)
    }
    lb = [(i, "body_mass_g") for i in penguins.index[IN_LB]]
    return {
        "cells correct": f"{len(right) / len(cells):.0%}",
        "weights in lb correct": f"{len(right & set(lb)) / len(lb):.0%}",
    }


def template_note(row) -> str:
    """One penguin, as a note that always follows the same template."""
    sex = row.sex if isinstance(row.sex, str) else "Sex unknown"
    if pd.isna(row.body_mass_g):
        return f"{row.species}, {row.island}. {sex}. Not measured."
    return (f"{row.species}, {row.island}. {sex}. Bill {row.bill_length_mm:g} mm long, "
            f"{row.bill_depth_mm:g} mm deep; flipper {row.flipper_length_mm:g} mm; mass {row.body_mass_g:g} g.")


templated_notes = pd.DataFrame({"note": [template_note(row) for row in penguins.itertuples()]})

## A data cleaning step in a larger program

Suppose you are writing a data analysis script for a penguin field study.
The script loads this season's field notes, turns them into a clean table,
and then analyses it. For the preprocessing step, you've written parsing logic to parse
the data out of the field notes, but every season you encounter a field note your logic
doesn't handle. Instead of rewriting your code every season, you wish your code
could just adapt itself when you run it, according to its inputs. JAZ's agent primitive
`invoke` is for exactly that: it is thought of as
**a function an LLM implements dynamically at runtime, based on the actual inputs it received**.

**[COMMENT] here we should show some examples of how complex is the peguin raw data**
<!-- There is one note per penguin, 344 in all, and they were written by different people in different ways: in mm, cm or
inches, in g, kg or lb, and with the sex sometimes given only as "she", "hen" or "♂":
Notes this varied are hard to parse with rules written in advance, but easy for a person (or an
LLM) to read. So we let an agent do this step of the script with JAZ's `invoke`: -->

In [ ]:
notes.head() # As you can see the data is unstructure and we want to extract the numerical values correctly
# We want extract tuple of (bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g (floats, in mm and g))

,note
0,"Adelie, Torgersen. Male. Bill 39.1 mm long, 18.7 mm deep; flipper 181 mm; mass 3750 g."
1,"Caught a female Adelie on Torgersen this morning. Her bill measured 39.5 mm by 17.4 mm, flipper 186 mm, and she weighed 3.8 kg."
2,"Torgersen — Adélie hen, flipper 195, bill 40.3 long x 18 deep, 3,250 g on the scale."
3,"Adelie on Torgersen, but it slipped away before we could measure or weigh it. Nothing recorded, sex unknown."
4,"She's an Adelie from Torgersen: culmen 36.7 mm, bill depth 19.3 mm, flipper length 193 mm, body mass 3450 g."


In [20]:
from jaz import invoke
from jaz.hooks import ReturnType

task = """\
Transcribe the field notes into a DataFrame. Rows are penguins.
Columns are species (Adelie, Chinstrap or Gentoo), island (Biscoe, Dream or Torgersen),
bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g (floats, in mm and g),
and sex (MALE or FEMALE).
"""

clean = invoke(
    # PrintTrajectory(abbrev_prefix_len=10, abbrev_suffix_len=5),  # uncomment to see the trajectory
    ReturnType(pd.DataFrame),
    task=task,
    notes=notes,
)

clean.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,NaN,NaN,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,NaN


In [21]:
score(clean) # This is unclear

{'cells correct': '88%', 'weights in lb correct': '0%'}

**[Comment] This needs to shrink to 4 bullets. We should provide links for more curious users to read more about framework as well comparisons**
The keyword arguments `task` and `notes` are the *inputs*: they become variables
`task` and `notes` in a Python REPL the LLM writes in, and are shown as sections
titled `<task>` and `<notes>` in the LLM's prompt. The names "task" and "notes" don't matter,
as long as the LLM understands what they mean. They're also not restricted to any particular
type — you can pass any Python object in your code as an input to `invoke`.

A few things of note:
- *There is no special argument for the task prompt.* It's just an input. Like any other input,
  it is shown in the LLM's prompt and made available as a variable in the REPL.
- **Programmatic inputs.** `notes` is the DataFrame object itself, so the LLM can write
  code to interact with it programmatically in the REPL, no matter how large it is. 
  There is no need to serialize it into the agent's prompt, or register MCP tools
  that allow the agent to query the database.
- **Programmatic output.** The LLM finishes by writing a return statement like
  in a function body, so `invoke`'s output can be any Python object the LLM wrote code
  to construct. In our example, the LLM constructed a DataFrame in its REPL and returned it,
  and now you can write code that directly uses the real DataFrame.
  `invoke`'s output is not restricted to a string or JSON that you'd then have to convert
  into a DataFrame.

You may have also noticed positional arguments to `invoke`: `ReturnType` and `PrintTrajectory`.
These are *hooks*, whose job is to observe or influence `invoke` as it runs. 

- *Hooks enforce constraints.* JAZ's built-in hook `ReturnType` enforces the output type,
  so that `ReturnType(pd.DataFrame)` guarantees that `clean` is a DataFrame.
  There are also hooks for budget constraints and validating the LLM's code.
- *Hooks provide observability and resumability.*
  JAZ's built-in hooks provide logging, tracing, and resuming from a checkpoint by
  replaying a log. Here, `PrintTrajectory` is a custom hook we wrote for this tutorial
  to show just the LLM responses and REPL outputs of a trajectory.

Now, although `invoke` correctly extracted around 90% of the field notes, it keeps
getting all the weights that were recorded in lb wrong.
Perhaps the team's notebook will give a clue why...

## Tools are just inputs that are callable

The field team kept a notebook during the season that's available through an API endpoint.
Let's give the agent a tool `view_notebook` that accesses it.

**[Comments] Make it an actual tool call based on web API call**

In [22]:
def view_notebook() -> str:
    """Returns the field team's notebook from the season the measurements were taken, which records
    known problems with them. Make sure to call this function and read its output before doing anything
    with the field data."""
    # Simulating the API call with hard-coded response
    return (
        "FIELD NOTEBOOK\n"
        "- Torgersen: rough landing; lost a caliper and switched to the spare (checked against the reference: OK).\n"
        "- Biscoe: rain most of the week. Some notes got wet and were recopied in camp.\n"
        "- Dream: back at the station we found that the visiting volunteer's spring scale had not been "
        "tared properly. Every weight they recorded in lb, on any island, reads 0.2 lb too high. Bill and "
        "flipper measurements are unaffected.\n"
        "- All notes were transcribed into the spreadsheet by hand at the station."
    )

Normally, giving an agent a tool requires writing a tool spec and registering it
on the agent or on an MCP server. But with `invoke`, having access to a tool
is just the same thing as having the `view_notebook` function available in the agent's
REPL. *So we can just pass `view_notebook` as a regular input (keyword argument) to `invoke`.*
JAZ automatically shows its signature and docstring to the LLM. There's no boilerplate.

In [23]:
clean = invoke(
    # PrintTrajectory(abbrev_prefix_len=10, abbrev_suffix_len=5),  # uncomment to see the trajectory
    ReturnType(pd.DataFrame),
    task=task,
    notes=notes,
    view_notebook=view_notebook,
)

score(clean)

{'cells correct': '94%', 'weights in lb correct': '100%'}

Having read the notebook, the agent now correctly subtracts 0.2 lb from every weight
given in lb before converting it to grams.
But there's still a few cells it's getting wrong.
So far the agent has been looking at a few example field notes and attempting to
write parsing logic for all notes — perhaps that's fundamentally the wrong approach,
and we should launch launch subagents that manually read the notes.

## What if one agent isn't reliable enough?

**[Comment] Task decomposition capability, wrong heading**

344 notes is a lot for a single agent to process, so it still makes mistakes. We can
improve the reliability by splitting up the notes into small chunks and
launch subagents that manually read each chunk to transcribe them by hand,
similar to [RLM](https://arxiv.org/abs/2512.24601)'s approach to processing large
pieces of text.

In JAZ, subagents (recursive `invoke`) are automatically available to every agent,
including subagents themselves. So all we have to do is to prompt `invoke` to use them.
The subagents need the
notebook too, though, and an input to `invoke` only reaches the agent it was passed to. So we move
`view_notebook` into `scope(...)`, which makes it an input to every agent inside the `with` block,
subagents included.

In [23]:
from jaz import scope

task_with_subagents = task + """
Split the notes up into multiple small chunks and launch parallel subagents
that read each chunk manually to transcribe by hand.
"""

with scope(view_notebook=view_notebook):
    clean = invoke(
        # PrintTrajectory(abbrev_prefix_len=10, abbrev_suffix_len=5),  # uncomment to see the trajectory
        ReturnType(pd.DataFrame),
        task=task_with_subagents,
        notes=notes,
    )

score(clean)

{'cells correct': '100%', 'weights in lb correct': '100%'}

Now all field notes are getting transcribed correctly!

## Hooks: logging and constraints
**[Comment] Change the section heading to something more engaging, it should be more like want see how jaz works in real time.... we are very transparent how agent works....**
`ReturnType` is one of many *hooks*. A hook observes the run through events and can change it, and
JAZ ships ready-made ones for logging, limits, budgets and validation. You can pass a hook
positionally, as we did with `ReturnType`, so that it applies to that one call, or use it as a context
manager, so that it applies to every `invoke` inside the block, subagents included.

Suppose that next season, the field team writes every note in the same template:

In [9]:
templated_notes.head()

,note
0,"Adelie, Torgersen. MALE. Bill 39.1 mm long, 18.7 mm deep; flipper 181 mm; mass 3750 g."
1,"Adelie, Torgersen. FEMALE. Bill 39.5 mm long, 17.4 mm deep; flipper 186 mm; mass 3800 g."
2,"Adelie, Torgersen. FEMALE. Bill 40.3 mm long, 18 mm deep; flipper 195 mm; mass 3250 g."
3,"Adelie, Torgersen. Sex unknown. Not measured."
4,"Adelie, Torgersen. FEMALE. Bill 36.7 mm long, 19.3 mm deep; flipper 193 mm; mass 3450 g."


Notes like these can be reliably parsed with regular expressions in a single piece of code,
so we no longer want the agent to take more than a couple of turns or spawn subagents.
We enforce that with
`IterationLimit(max_iterations=5)` and `RecursionLimit(max_depth=1)`, and we watch what happens with
`PrintLogger`, which logs every step of the run:

**[Comment] The print output is huge, we should type out how the pretty print will look like in the markdown with short or hypothetical example.***

In [ ]:
from jaz.hooks import IterationLimit, PrintLogger, RecursionLimit

with PrintLogger(), IterationLimit(max_iterations=5), RecursionLimit(max_depth=1):
    templated = invoke(
        ReturnType(pd.DataFrame),
        task=task,
        notes=templated_notes,
    )

score(templated)

2026-11-26 00:10:13 [Agent] invoke enter: inputs={'task': 'Transcribe the field notes into a DataFrame. Rows are penguins.\nColumns are species (Adelie, Chinstrap or Gentoo), island (Biscoe, Dream or Torgersen),\nbill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g (floats, in mm and g),\nand sex (MALE or FEMALE).\n', 'notes':                                                                                          note
0      Adelie, Torgersen. MALE. Bill 39.1 mm long, 18.7 mm deep; flipper 181 mm; mass 3750 g.
1    Adelie, Torgersen. FEMALE. Bill 39.5 mm long, 17.4 mm deep; flipper 186 mm; mass 3800 g.
2      Adelie, Torgersen. FEMALE. Bill 40.3 mm long, 18 mm deep; flipper 195 mm; mass 3250 g.
3                                               Adelie, Torgersen. Sex unknown. Not measured.
4    Adelie, Torgersen. FEMALE. Bill 36.7 mm long, 19.3 mm deep; flipper 193 mm; mass 3450 g.
..                                                                                        ...
339 

{'cells correct': '100%', 'weights in lb correct': '100%'}

You can now see the prompt and the full agent trajectory, including all LLM responses
and REPL outputs. The agent probably finished within 5 turns, and if didn't,
`IterationLimit` would have stopped it with an `IterationLimitExhaustedError`.
The agent didn't spawn any subagents, and in fact could not, as `RecursionLimit(max_depth=1)`
makes subagents unavailable to the root `invoke`.

You've probably noticed that we applied some hooks as a `with` block and some as
positional arguments to `invoke`. The difference is akin to the difference between
`scope(...)` and a regular keyword argument we saw earlier: a context manager hook
applies to all `invoke`s (including recursive subagents) under the `with` block,
whereas a positional argument hook applies only to the `invoke` it's passed to
and not to its subagents. Thus, `ReturnType` is typically a positional argument,
whereas limits and loggers are typically context managers, but that's not necessarily
always the case.

## Configuration: imports and models

With the data extracted, your script now moves on to data analysis:
how much heavier does a penguin get for
each extra millimetre of flipper, in each species? A few errors may have survived the cleaning, so
we want a robust regression, such as the Theil–Sen estimator, which ignores a few outliers.
[SciPy](https://scipy.org) already implements it, and the task is easy, so a smaller model will do.

Both are *configuration*. By default the agent's REPL can't import anything, and every agent uses
the model set with `jaz.configure` in the setup. `ConfigOverride` changes the configuration for one
call when passed positionally, like a hook (or for a whole block, used as a context manager). Here
it allows the REPL to import `scipy` and switches to `gpt-5.4-nano`:

**[COMMENT] I actually feel like this should be the first example because it kind of shows the quik start at once.**
**So we should start with a quick start problem and then move to a uniform penguin problem**

In [ ]:
from jaz import ConfigOverride

slopes = invoke(
    ConfigOverride(  # can also use as a `with` block
        llm=LiteLLM(model="openai/gpt-5.4-nano"),
        repl=PythonREPL(allowed_imports=["scipy"]),
    ),
    # PrintTrajectory(abbrev_prefix_len=10, abbrev_suffix_len=5),  # uncomment to see the trajectory
    ReturnType(dict[str, float]),
    task=(
        "For each species, fit a robust linear regression of body mass on flipper length with "
        "scipy.stats.theilslopes, and return the slopes in grams per mm, keyed by species."
    ),
    penguins=clean,
)

slopes

{'Adelie': 34.09090909090909,
 'Chinstrap': 34.091775093839075,
 'Gentoo': 56.800559297872205}

## Summary

- **Inputs.** `invoke(**inputs)` takes any Python objects as keyword arguments: the prompt, data
  such as a DataFrame, and tools such as `view_notebook` all go in the same way. Each becomes a
  variable in the agent's REPL and is shown in its prompt.
- **Outputs.** The agent returns a real Python object, which you can type with `ReturnType`.
- **Subagents and scope.** Any agent can call `invoke` to delegate work to subagents.
  `with scope(...)` makes inputs, such as tools, available to every agent in the tree.
- **Hooks** observe and constrain a run: `PrintLogger`, `IterationLimit`, `RecursionLimit`,
  `ReturnType` and more, positionally for one call or with `with` for a block.
- **Configuration** sets the model and the REPL's sandbox, globally with `jaz.configure` or for one
  call or block with `ConfigOverride`.